# 시드 문맥 감성 변화 방향 조합 분석

## 목적

`v_2_2_seed_context_sentiment_controls.ipynb`의 변화량 데이터(`change_df`)를 사용해, 긍정 문맥 변화와 부정 문맥 변화가 서로 섞여서 ESG 등급 변화와의 관계가 약하게 보였는지 확인한다.

핵심 아이디어는 선형 회귀식 하나에 긍정 변화와 부정 변화를 넣는 대신, 변화 방향 조합을 직접 만든다.

```text
긍정 증가 / 부정 감소
긍정 증가 / 부정 증가
긍정 감소 / 부정 감소
긍정 감소 / 부정 증가
```

이 조합별로 다음을 비교한다.

- ESG 등급 변화 평균
- ESG 등급 상승 비율
- ESG 등급 하락 비율
- 연도별 패턴
- Kruskal-Wallis 및 카이제곱 검정

## 입력

- `final/v_2_2_seed_context_sentiment_controls_change.csv`

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import kruskal, chi2_contingency
import statsmodels.api as sm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

LOCAL_ROOT = Path.cwd()
ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/My Drive/UD_26"),
    LOCAL_ROOT,
    LOCAL_ROOT.parent,
]


def first_existing(candidates, default=None):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return default if default is not None else candidates[0]


ROOT = first_existing(
    [p for p in ROOT_CANDIDATES if (p / "data").exists() or (p / "final").exists()],
    LOCAL_ROOT,
)
FINAL_DIR = ROOT / "final"
INPUT_PATH = first_existing([
    FINAL_DIR / "v_2_2_seed_context_sentiment_controls_change.csv",
    LOCAL_ROOT / "final" / "v_2_2_seed_context_sentiment_controls_change.csv",
    LOCAL_ROOT / "v_2_2_seed_context_sentiment_controls_change.csv",
])

print("ROOT:", ROOT)
print("INPUT_PATH:", INPUT_PATH, "|", "OK" if INPUT_PATH.exists() else "MISSING")

ROOT: /content/drive/MyDrive/UD_26
INPUT_PATH: /content/drive/MyDrive/UD_26/final/v_2_2_seed_context_sentiment_controls_change.csv | OK


In [2]:
change_df = pd.read_csv(INPUT_PATH, dtype={"stock_code": "string"}, encoding="utf-8-sig")
change_df["stock_code"] = change_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
for col in ["fiscal_year", "esg_year", "delta_esg_grade_num", "esg_grade_num", "lag_esg_grade_num"]:
    if col in change_df.columns:
        change_df[col] = pd.to_numeric(change_df[col], errors="coerce")

print("change_df:", change_df.shape)
display(change_df.groupby(["fiscal_year", "esg_year", "grade_change_group"]).size().rename("n").reset_index())
display(change_df.head())

change_df: (252, 102)


,fiscal_year,esg_year,grade_change_group,n
0,2023,2024,decrease,23
1,2023,2024,increase,34
2,2023,2024,no_change,69
3,2024,2025,decrease,30
4,2024,2025,increase,23
5,2024,2025,no_change,73


,stock_code,company_name,fiscal_year,esg_year,rcept_no,total_word_count,total_char_count,section_count,seed_sentence_count,positive_seed_sentence_count,negative_seed_sentence_count,neutral_seed_sentence_count,mean_seed_sentence_sentiment,unique_seed_terms_in_sentences,seed_term_context_count,seed_term_occurrence_count,positive_seed_term_context_count,negative_seed_term_context_count,neutral_seed_term_context_count,mean_seed_term_context_sentiment,unique_seed_terms_matched,positive_seed_sentence_share,negative_seed_sentence_share,neutral_seed_sentence_share,positive_seed_term_context_share,negative_seed_term_context_share,seed_sentence_per_1000_words,seed_term_context_per_1000_words,seed_term_occurrence_per_1000_words,industry,esg_grade,e_grade,s_grade,g_grade,esg_grade_num,e_grade_num,s_grade_num,g_grade_num,net_seed_sentence_count,net_seed_sentence_share,net_seed_term_context_count,net_seed_term_context_share,positive_seed_sentence_per_1000_words,negative_seed_sentence_per_1000_words,positive_seed_term_context_per_1000_words,negative_seed_term_context_per_1000_words,c_positive_seed_sentence_share,c_negative_seed_sentence_share,c_positive_seed_term_context_share,c_negative_seed_term_context_share,c_positive_seed_sentence_count,c_negative_seed_sentence_count,sentence_share_interaction,term_context_share_interaction,sentence_count_interaction,positive_sentence_group,negative_sentence_group,pos_neg_sentence_group,positive_context_group,negative_context_group,pos_neg_context_group,lag_esg_grade_num,delta_esg_grade_num,lag_positive_seed_sentence_count,delta_positive_seed_sentence_count,lag_negative_seed_sentence_count,delta_negative_seed_sentence_count,lag_net_seed_sentence_count,delta_net_seed_sentence_count,lag_positive_seed_sentence_share,delta_positive_seed_sentence_share,lag_negative_seed_sentence_share,delta_negative_seed_sentence_share,lag_net_seed_sentence_share,delta_net_seed_sentence_share,lag_mean_seed_sentence_sentiment,delta_mean_seed_sentence_sentiment,lag_positive_seed_term_context_count,delta_positive_seed_term_context_count,lag_negative_seed_term_context_count,delta_negative_seed_term_context_count,lag_net_seed_term_context_count,delta_net_seed_term_context_count,lag_positive_seed_term_context_share,delta_positive_seed_term_context_share,lag_negative_seed_term_context_share,delta_negative_seed_term_context_share,lag_net_seed_term_context_share,delta_net_seed_term_context_share,lag_mean_seed_term_context_sentiment,delta_mean_seed_term_context_sentiment,lag_seed_sentence_count,delta_seed_sentence_count,lag_seed_term_context_count,delta_seed_term_context_count,lag_unique_seed_terms_matched,delta_unique_seed_terms_matched,lag_total_word_count,delta_total_word_count,lag_fiscal_year,year_gap,grade_change_group
0,000020,동화약품,2023,2024,20240319000652,8065,38467,3,35,2,1,32,0.036592,13,67,114,2,2,63,0.008703,13,0.057143,0.028571,0.914286,0.029851,0.029851,4.339740,8.307502,14.135152,NaN,C,B,B,C,1,2,2,1,1,0.028571,0,0.000000,0.247985,0.123993,0.247985,0.247985,-0.026795,-0.000641,-0.033488,0.006303,-7.214286,-1.775132,0.000017,-0.000211,12.806311,low,high,pos_low__neg_high,low,high,pos_low__neg_high,1.0,0.0,2.0,0.0,2.0,-1.0,0.0,1.0,0.057143,0.000000,0.057143,-0.028571,0.000000,0.028571,0.018317,0.018276,2.0,0.0,3.0,-1.0,-1.0,1.0,0.029412,0.000439,0.044118,-0.014267,-0.014706,0.014706,-0.000832,0.009535,35.0,0.0,68.0,-1.0,14.0,-1.0,7808.0,257.0,2022.0,1.0,no_change
1,000020,동화약품,2024,2025,20250318000739,8097,39259,3,37,2,0,35,0.053468,14,68,115,2,0,66,0.029093,14,0.054054,0.000000,0.945946,0.029412,0.000000,4.569594,8.398172,14.202791,NaN,C,B,C,C,1,2,1,1,2,0.054054,2,0.029412,0.247005,0.000000,0.247005,0.000000,-0.029884,-0.029213,-0.033927,-0.023547,-7.214286,-2.775132,0.000873,0.000799,20.020597,low,low,pos_low__neg_low,low,low,pos_low__neg_low,1.0,0.0,2.0,0.0,1.0,-1.0,1.0,1.0,0.057143,-0.003089,0.028571,-0.028571,0.028571,0.025483,0.036592,0.016876,2.0,0.0,2.0,-2.0,0.0,2.0,0.029851,-0.000439,0.029851,-0.029851,0.000000,0.0

In [3]:
# Direction-group construction.
# Use >0 as increase, <0 as decrease, and 0 as no_change.
# For a stricter robustness check, an epsilon threshold can be set above zero.

EPSILON = 0.0


def direction_label(value, epsilon=EPSILON):
    if pd.isna(value):
        return "missing"
    if value > epsilon:
        return "up"
    if value < -epsilon:
        return "down"
    return "flat"


def make_direction_groups(df, positive_col, negative_col, prefix):
    out = df.copy()
    out[f"{prefix}_positive_direction"] = out[positive_col].map(direction_label)
    out[f"{prefix}_negative_direction"] = out[negative_col].map(direction_label)
    out[f"{prefix}_direction_group"] = (
        "pos_" + out[f"{prefix}_positive_direction"] + "__neg_" + out[f"{prefix}_negative_direction"]
    )
    out[f"{prefix}_favorable_change"] = (
        out[f"{prefix}_positive_direction"].eq("up") & out[f"{prefix}_negative_direction"].isin(["down", "flat"])
    ).astype(int)
    out[f"{prefix}_unfavorable_change"] = (
        out[f"{prefix}_positive_direction"].isin(["down", "flat"]) & out[f"{prefix}_negative_direction"].eq("up")
    ).astype(int)
    out[f"{prefix}_mixed_up"] = (
        out[f"{prefix}_positive_direction"].eq("up") & out[f"{prefix}_negative_direction"].eq("up")
    ).astype(int)
    out[f"{prefix}_mixed_down"] = (
        out[f"{prefix}_positive_direction"].eq("down") & out[f"{prefix}_negative_direction"].eq("down")
    ).astype(int)
    return out


change_df = make_direction_groups(
    change_df,
    "delta_positive_seed_sentence_share",
    "delta_negative_seed_sentence_share",
    "sentence_share",
)
change_df = make_direction_groups(
    change_df,
    "delta_positive_seed_sentence_count",
    "delta_negative_seed_sentence_count",
    "sentence_count",
)
change_df = make_direction_groups(
    change_df,
    "delta_positive_seed_term_context_share",
    "delta_negative_seed_term_context_share",
    "context_share",
)

for col in ["sentence_share_direction_group", "sentence_count_direction_group", "context_share_direction_group"]:
    print(col)
    display(change_df[col].value_counts(dropna=False).rename("n"))

sentence_share_direction_group


,n
sentence_share_direction_group,
pos_down__neg_up,56
pos_up__neg_down,54
pos_up__neg_up,48
pos_down__neg_down,45
pos_down__neg_flat,18
pos_flat__neg_flat,11
pos_up__neg_flat,11
pos_flat__neg_up,5
pos_flat__neg_down,4


sentence_count_direction_group


,n
sentence_count_direction_group,
pos_flat__neg_flat,40
pos_down__neg_up,39
pos_up__neg_flat,35
pos_up__neg_down,34
pos_down__neg_flat,28
pos_up__neg_up,24
pos_down__neg_down,22
pos_flat__neg_up,18
pos_flat__neg_down,12


context_share_direction_group


,n
context_share_direction_group,
pos_down__neg_up,61
pos_up__neg_down,54
pos_down__neg_down,53
pos_up__neg_up,39
pos_down__neg_flat,19
pos_up__neg_flat,9
pos_flat__neg_flat,9
pos_flat__neg_down,4
pos_flat__neg_up,4


In [4]:
def summarize_direction_group(df, group_col):
    summary = (
        df.groupby(group_col)
        .agg(
            n=("delta_esg_grade_num", "count"),
            mean_delta_grade=("delta_esg_grade_num", "mean"),
            median_delta_grade=("delta_esg_grade_num", "median"),
            grade_up_rate=("delta_esg_grade_num", lambda s: float((s > 0).mean())),
            grade_down_rate=("delta_esg_grade_num", lambda s: float((s < 0).mean())),
            grade_no_change_rate=("delta_esg_grade_num", lambda s: float((s == 0).mean())),
            mean_delta_pos_share=("delta_positive_seed_sentence_share", "mean"),
            mean_delta_neg_share=("delta_negative_seed_sentence_share", "mean"),
            mean_delta_net_share=("delta_net_seed_sentence_share", "mean"),
        )
        .reset_index()
        .sort_values("mean_delta_grade", ascending=False)
    )
    return summary

sentence_share_group_summary = summarize_direction_group(change_df, "sentence_share_direction_group")
sentence_count_group_summary = summarize_direction_group(change_df, "sentence_count_direction_group")
context_share_group_summary = summarize_direction_group(change_df, "context_share_direction_group")

print("Sentence-share direction groups")
display(sentence_share_group_summary)
print("Sentence-count direction groups")
display(sentence_count_group_summary)
print("Term-context-share direction groups")
display(context_share_group_summary)

Sentence-share direction groups


,sentence_share_direction_group,n,mean_delta_grade,median_delta_grade,grade_up_rate,grade_down_rate,grade_no_change_rate,mean_delta_pos_share,mean_delta_neg_share,mean_delta_net_share
5,pos_flat__neg_up,5,0.400000,0.0,0.400000,0.000000,0.600000,0.000000,0.015915,-0.015915
3,pos_flat__neg_down,4,0.250000,0.0,0.250000,0.000000,0.750000,0.000000,-0.013641,0.013641
0,pos_down__neg_down,45,0.222222,0.0,0.288889,0.155556,0.555556,-0.022283,-0.012899,-0.009383
7,pos_up__neg_flat,11,0.181818,0.0,0.272727,0.090909,0.636364,0.025226,0.000000,0.025226
8,pos_up__neg_up,48,0.166667,0.0,0.250000,0.125000,0.625000,0.022607,0.012816,0.009790
1,pos_down__neg_flat,18,0.055556,0.0,0.277778,0.277778,0.444444,-0.017162,0.000000,-0.017162
4,pos_flat__neg_flat,11,0.000000,0.0,0.090909,0.090909,0.818182,0.000000,0.000000,0.000000
2,pos_down__neg_up,56,-0.107143,0.0,0.196429,0.267857,0.535714,-0.031290,0.018653,-0.049943
6,pos_up__neg_down,54,-0.166667,0.0,0.166667,0.333333,0.500000,0.026437,-0.017883,0.044320


Sentence-count direction groups


,sentence_count_direction_group,n,mean_delta_grade,median_delta_grade,grade_up_rate,grade_down_rate,grade_no_change_rate,mean_delta_pos_share,mean_delta_neg_share,mean_delta_net_share
4,pos_flat__neg_flat,40,0.225000,0.0,0.250000,0.075000,0.675000,-0.002353,-0.000274,-0.002079
8,pos_up__neg_up,24,0.208333,0.0,0.291667,0.125000,0.583333,0.032464,0.016134,0.016330
0,pos_down__neg_down,22,0.136364,0.0,0.318182,0.272727,0.409091,-0.032792,-0.016178,-0.016614
3,pos_flat__neg_down,12,0.083333,0.0,0.166667,0.250000,0.583333,-0.001169,-0.022073,0.020904
7,pos_up__neg_flat,35,0.028571,0.0,0.171429,0.171429,0.657143,0.028814,-0.000597,0.029411
5,pos_flat__neg_up,18,0.000000,0.0,0.222222,0.166667,0.611111,-0.004095,0.017505,-0.021600
2,pos_down__neg_up,39,0.000000,0.0,0.230769,0.205128,0.564103,-0.033345,0.024658,-0.058003
1,pos_down__neg_flat,28,-0.071429,0.0,0.250000,0.321429,0.428571,-0.027516,0.000323,-0.027839
6,pos_up__neg_down,34,-0.235294,0.0,0.147059,0.352941,0.500000,0.026849,-0.025934,0.052783


Term-context-share direction groups


,context_share_direction_group,n,mean_delta_grade,median_delta_grade,grade_up_rate,grade_down_rate,grade_no_change_rate,mean_delta_pos_share,mean_delta_neg_share,mean_delta_net_share
5,pos_flat__neg_up,4,0.500000,0.5,0.500000,0.000000,0.500000,0.000000,0.011570,-0.011570
3,pos_flat__neg_down,4,0.250000,0.0,0.250000,0.000000,0.750000,0.000000,-0.005987,0.005987
7,pos_up__neg_flat,9,0.111111,0.0,0.222222,0.111111,0.666667,0.031655,-0.000056,0.031711
4,pos_flat__neg_flat,9,0.111111,0.0,0.222222,0.111111,0.666667,0.000265,0.000000,0.000265
8,pos_up__neg_up,39,0.102564,0.0,0.179487,0.076923,0.743590,0.023008,0.011497,0.011511
0,pos_down__neg_down,53,0.056604,0.0,0.188679,0.188679,0.622642,-0.018730,-0.010947,-0.007783
1,pos_down__neg_flat,19,0.052632,0.0,0.263158,0.263158,0.473684,-0.016259,0.000000,-0.016259
2,pos_down__neg_up,61,0.000000,0.0,0.278689,0.262295,0.459016,-0.026023,0.018838,-0.044862
6,pos_up__neg_down,54,-0.074074,0.0,0.203704,0.314815,0.481481,0.026494,-0.016677,0.043172


In [5]:
def kruskal_by_group(df, group_col, value_col="delta_esg_grade_num"):
    groups = [g[value_col].dropna().values for _, g in df.groupby(group_col) if len(g[value_col].dropna()) > 0]
    if len(groups) < 2:
        return {"group_col": group_col, "test": "Kruskal-Wallis", "statistic": np.nan, "p_value": np.nan}
    stat, p_value = kruskal(*groups)
    return {"group_col": group_col, "test": "Kruskal-Wallis", "statistic": float(stat), "p_value": float(p_value)}


def chi_square_grade_change(df, group_col):
    table = pd.crosstab(df[group_col], df["grade_change_group"])
    if table.shape[0] < 2 or table.shape[1] < 2:
        return {"group_col": group_col, "test": "chi-square", "statistic": np.nan, "p_value": np.nan, "dof": np.nan}
    stat, p_value, dof, expected = chi2_contingency(table)
    return {"group_col": group_col, "test": "chi-square", "statistic": float(stat), "p_value": float(p_value), "dof": int(dof)}


test_rows = []
for group_col in ["sentence_share_direction_group", "sentence_count_direction_group", "context_share_direction_group"]:
    test_rows.append(kruskal_by_group(change_df, group_col))
    test_rows.append(chi_square_grade_change(change_df, group_col))

group_test_df = pd.DataFrame(test_rows).sort_values(["group_col", "test"])
display(group_test_df)

for group_col in ["sentence_share_direction_group", "sentence_count_direction_group", "context_share_direction_group"]:
    print("\n", group_col)
    display(pd.crosstab(change_df[group_col], change_df["grade_change_group"], margins=True))

,group_col,test,statistic,p_value,dof
4,context_share_direction_group,Kruskal-Wallis,4.935352,0.764463,NaN
5,context_share_direction_group,chi-square,18.091260,0.318580,16.0
2,sentence_count_direction_group,Kruskal-Wallis,8.402891,0.395136,NaN
3,sentence_count_direction_group,chi-square,16.812250,0.397846,16.0
0,sentence_share_direction_group,Kruskal-Wallis,11.050432,0.198858,NaN
1,sentence_share_direction_group,chi-square,17.567914,0.349798,16.0



 sentence_share_direction_group


grade_change_group,decrease,increase,no_change,All
sentence_share_direction_group,,,,
pos_down__neg_down,7,13,25,45
pos_down__neg_flat,5,5,8,18
pos_down__neg_up,15,11,30,56
pos_flat__neg_down,0,1,3,4
pos_flat__neg_flat,1,1,9,11
pos_flat__neg_up,0,2,3,5
pos_up__neg_down,18,9,27,54
pos_up__neg_flat,1,3,7,11
pos_up__neg_up,6,12,30,48



 sentence_count_direction_group


grade_change_group,decrease,increase,no_change,All
sentence_count_direction_group,,,,
pos_down__neg_down,6,7,9,22
pos_down__neg_flat,9,7,12,28
pos_down__neg_up,8,9,22,39
pos_flat__neg_down,3,2,7,12
pos_flat__neg_flat,3,10,27,40
pos_flat__neg_up,3,4,11,18
pos_up__neg_down,12,5,17,34
pos_up__neg_flat,6,6,23,35
pos_up__neg_up,3,7,14,24



 context_share_direction_group


grade_change_group,decrease,increase,no_change,All
context_share_direction_group,,,,
pos_down__neg_down,10,10,33,53
pos_down__neg_flat,5,5,9,19
pos_down__neg_up,16,17,28,61
pos_flat__neg_down,0,1,3,4
pos_flat__neg_flat,1,2,6,9
pos_flat__neg_up,0,2,2,4
pos_up__neg_down,17,11,26,54
pos_up__neg_flat,1,2,6,9
pos_up__neg_up,3,7,29,39


In [6]:
# Compact favorable/unfavorable comparison.
# favorable: positive up and negative down/flat
# unfavorable: positive down/flat and negative up

comparison_rows = []
for prefix in ["sentence_share", "sentence_count", "context_share"]:
    group_col = f"{prefix}_direction_bucket"
    favorable_col = f"{prefix}_favorable_change"
    unfavorable_col = f"{prefix}_unfavorable_change"

    conditions = [
        change_df[favorable_col].eq(1),
        change_df[unfavorable_col].eq(1),
    ]
    choices = ["favorable", "unfavorable"]
    change_df[group_col] = np.select(conditions, choices, default="mixed_or_flat")

    summary = (
        change_df.groupby(group_col)
        .agg(
            n=("delta_esg_grade_num", "count"),
            mean_delta_grade=("delta_esg_grade_num", "mean"),
            median_delta_grade=("delta_esg_grade_num", "median"),
            grade_up_rate=("delta_esg_grade_num", lambda s: float((s > 0).mean())),
            grade_down_rate=("delta_esg_grade_num", lambda s: float((s < 0).mean())),
        )
        .reset_index()
    )
    summary.insert(0, "prefix", prefix)
    comparison_rows.append(summary)

favorable_summary_df = pd.concat(comparison_rows, ignore_index=True)
display(favorable_summary_df.sort_values(["prefix", "mean_delta_grade"], ascending=[True, False]))

bucket_test_rows = []
for group_col in ["sentence_share_direction_bucket", "sentence_count_direction_bucket", "context_share_direction_bucket"]:
    bucket_test_rows.append(kruskal_by_group(change_df, group_col))
    bucket_test_rows.append(chi_square_grade_change(change_df, group_col))

bucket_test_df = pd.DataFrame(bucket_test_rows).sort_values(["group_col", "test"])
display(bucket_test_df)

,prefix,sentence_share_direction_bucket,n,mean_delta_grade,median_delta_grade,grade_up_rate,grade_down_rate,sentence_count_direction_bucket,context_share_direction_bucket
7,context_share,NaN,124,0.080645,0.0,0.201613,0.153226,NaN,mixed_or_flat
8,context_share,NaN,65,0.030769,0.0,0.292308,0.246154,NaN,unfavorable
6,context_share,NaN,63,-0.047619,0.0,0.206349,0.285714,NaN,favorable
4,sentence_count,NaN,126,0.126984,0.0,0.261905,0.190476,mixed_or_flat,NaN
5,sentence_count,NaN,57,0.000000,0.0,0.228070,0.192982,unfavorable,NaN
3,sentence_count,NaN,69,-0.101449,0.0,0.159420,0.260870,favorable,NaN
1,sentence_share,mixed_or_flat,126,0.158730,0.0,0.253968,0.150794,NaN,NaN
2,sentence_share,unfavorable,61,-0.065574,0.0,0.213115,0.245902,NaN,NaN
0,sentence_share,favorable,65,-0.107692,0.0,0.184615,0.292308,NaN,NaN


,group_col,test,statistic,p_value,dof
4,context_share_direction_bucket,Kruskal-Wallis,1.522130,0.467169,NaN
5,context_share_direction_bucket,chi-square,8.730336,0.068205,4.0
2,sentence_count_direction_bucket,Kruskal-Wallis,3.382462,0.184292,NaN
3,sentence_count_direction_bucket,chi-square,3.339821,0.502648,4.0
0,sentence_share_direction_bucket,Kruskal-Wallis,5.384029,0.067744,NaN
1,sentence_share_direction_bucket,chi-square,6.008756,0.198495,4.0


In [7]:
# Year-transition-specific summaries.

yearly_summaries = []
for (fiscal_year, esg_year), year_df in change_df.groupby(["fiscal_year", "esg_year"]):
    for group_col in ["sentence_share_direction_group", "sentence_count_direction_group", "context_share_direction_group"]:
        tmp = summarize_direction_group(year_df, group_col)
        tmp.insert(0, "fiscal_year", fiscal_year)
        tmp.insert(1, "esg_year", esg_year)
        tmp.insert(2, "group_type", group_col)
        yearly_summaries.append(tmp)

yearly_group_summary_df = pd.concat(yearly_summaries, ignore_index=True)
display(yearly_group_summary_df.sort_values(["fiscal_year", "group_type", "mean_delta_grade"], ascending=[True, True, False]))

,fiscal_year,esg_year,group_type,sentence_share_direction_group,n,mean_delta_grade,median_delta_grade,grade_up_rate,grade_down_rate,grade_no_change_rate,mean_delta_pos_share,mean_delta_neg_share,mean_delta_net_share,sentence_count_direction_group,context_share_direction_group
18,2023,2024,context_share_direction_group,NaN,5,0.400000,0.0,0.400000,0.000000,0.600000,0.027940,0.000000,0.027940,NaN,pos_up__neg_flat
19,2023,2024,context_share_direction_group,NaN,3,0.333333,0.0,0.333333,0.000000,0.666667,0.000794,0.000000,0.000794,NaN,pos_flat__neg_flat
20,2023,2024,context_share_direction_group,NaN,26,0.153846,0.0,0.269231,0.153846,0.576923,-0.017863,-0.009230,-0.008633,NaN,pos_down__neg_down
21,2023,2024,context_share_direction_group,NaN,27,0.111111,0.0,0.296296,0.222222,0.481481,0.030065,-0.016892,0.046957,NaN,pos_up__neg_down
22,2023,2024,context_share_direction_group,NaN,11,0.090909,0.0,0.363636,0.363636,0.272727,-0.014940,0.000000,-0.014940,NaN,pos_down__neg_flat
23,2023,2024,context_share_direction_group,NaN,23,0.086957,0.0,0.173913,0.086957,0.739130,0.025420,0.009555,0.015866,NaN,pos_up__neg_up
24,2023,2024,context_share_direction_group,NaN,27,0.074074,0.0,0.296296,0.259259,0.444444,-0.031061,0.024474,-0.055535,NaN,pos_down__neg_up
25,2023,2024,context_share_direction_group,NaN,3,0.000000,0.0,0.000000,0.000000,1.000000,0.000000,-0.004038,0.004038,NaN,pos_flat__neg_down
26,2023,2024,context_share_direction_group,NaN,1,0.000000,0.0,0.000000,0.000000,1.000000,0.000000,0.026190,-0.026190,NaN,pos_flat__neg_up
9,2023,2024,sentence_count_direction_group,NaN,5,0.400000,0.0,0.200000,0.000000,0.800000,0.000260,-0.020903,0.021163,pos_flat__neg_down,NaN


In [8]:
# Optional compact regressions using direction buckets.
# These are easier to interpret than continuous delta regressions, but still exploratory.

def zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std


def fit_bucket_model(df, bucket_col):
    model_df = df[["delta_esg_grade_num", bucket_col, "delta_seed_sentence_count", "delta_total_word_count"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    dummies = pd.get_dummies(model_df[bucket_col], prefix=bucket_col, drop_first=True, dtype=float)
    X = pd.concat([
        dummies,
        model_df[["delta_seed_sentence_count", "delta_total_word_count"]].apply(zscore),
    ], axis=1)
    X = sm.add_constant(X, has_constant="add")
    y = model_df["delta_esg_grade_num"].astype(float)
    model = sm.OLS(y, X).fit(cov_type="HC3")
    return pd.DataFrame({
        "variable": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "p_value": model.pvalues.values,
        "r2": model.rsquared,
        "n": int(model.nobs),
    })

bucket_model_frames = []
for bucket_col in ["sentence_share_direction_bucket", "sentence_count_direction_bucket", "context_share_direction_bucket"]:
    frame = fit_bucket_model(change_df, bucket_col)
    frame.insert(0, "bucket_col", bucket_col)
    bucket_model_frames.append(frame)

bucket_model_df = pd.concat(bucket_model_frames, ignore_index=True)
display(bucket_model_df)

,bucket_col,variable,coef,std_err,p_value,r2,n
0,sentence_share_direction_bucket,const,-0.108851,0.095120,0.252480,0.028066,252
1,sentence_share_direction_bucket,sentence_share_direction_bucket_mixed_or_flat,0.265366,0.116104,0.022278,0.028066,252
2,sentence_share_direction_bucket,sentence_share_direction_bucket_unfavorable,0.049087,0.143480,0.732263,0.028066,252
3,sentence_share_direction_bucket,delta_seed_sentence_count,-0.001651,0.080640,0.983665,0.028066,252
4,sentence_share_direction_bucket,delta_total_word_count,0.035653,0.069919,0.610107,0.028066,252
5,sentence_count_direction_bucket,const,-0.105062,0.086739,0.225798,0.019712,252
6,sentence_count_direction_bucket,sentence_count_direction_bucket_mixed_or_flat,0.234024,0.112253,0.037089,0.019712,252
7,sentence_count_direction_bucket,sentence_count_direction_bucket_unfavorable,0.105065,0.131097,0.422882,0.019712,252
8,sentence_count_direction_bucket,delta_seed_sentence_count,0.006341,0.085237,0.940695,0.019712,252
9,sentence_count_direction_bucket,delta_total_word_count,0.041577,0.069327,0.548690,0.019712,252


In [9]:
OUTPUT_ANALYSIS_PATH = FINAL_DIR / "v_2_3_seed_context_sentiment_change_groups.csv"
OUTPUT_GROUP_SUMMARY_PATH = FINAL_DIR / "v_2_3_seed_context_sentiment_change_group_summary.csv"
OUTPUT_GROUP_TEST_PATH = FINAL_DIR / "v_2_3_seed_context_sentiment_change_group_tests.csv"
OUTPUT_YEARLY_SUMMARY_PATH = FINAL_DIR / "v_2_3_seed_context_sentiment_change_group_yearly_summary.csv"

change_df.to_csv(OUTPUT_ANALYSIS_PATH, index=False, encoding="utf-8-sig")
favorable_summary_df.to_csv(OUTPUT_GROUP_SUMMARY_PATH, index=False, encoding="utf-8-sig")
pd.concat([group_test_df, bucket_test_df], ignore_index=True).to_csv(OUTPUT_GROUP_TEST_PATH, index=False, encoding="utf-8-sig")
yearly_group_summary_df.to_csv(OUTPUT_YEARLY_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("saved analysis:", OUTPUT_ANALYSIS_PATH, change_df.shape)
print("saved group summary:", OUTPUT_GROUP_SUMMARY_PATH, favorable_summary_df.shape)
print("saved group tests:", OUTPUT_GROUP_TEST_PATH)
print("saved yearly summary:", OUTPUT_YEARLY_SUMMARY_PATH, yearly_group_summary_df.shape)

saved analysis: /content/drive/MyDrive/UD_26/final/v_2_3_seed_context_sentiment_change_groups.csv (252, 126)
saved group summary: /content/drive/MyDrive/UD_26/final/v_2_3_seed_context_sentiment_change_group_summary.csv (9, 9)
saved group tests: /content/drive/MyDrive/UD_26/final/v_2_3_seed_context_sentiment_change_group_tests.csv
saved yearly summary: /content/drive/MyDrive/UD_26/final/v_2_3_seed_context_sentiment_change_group_yearly_summary.csv (54, 15)
